In [8]:
# Cell 1: Install and Initialize the NLP Pipeline
!pip install transformers

from transformers import pipeline
import random

# Initialize a powerful zero-shot classification pipeline
# This will automatically download a lightweight model perfect for Colab
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Define the intents and mapping for VocloneTranslate
intents_map = {
    "greeting": {
        "label": "greeting or hello",
        "responses": [
            "Hello! Welcome to VocloneTranslate Support. How can I assist you with voice cloning or translation today?",
            "Hi there! What can I help you build or translate today?"
        ]
    },
    "translate": {
        "label": "language translation or audio translation text features",
        "responses": [
            "VocloneTranslate allows you to seamlessly translate audio and text across 20+ languages while preserving speech context.",
            "To translate, simply upload your source file, choose your target language, and hit Translate!"
        ]
    },
    "voice_cloning": {
        "label": "voice cloning or copying voice speech synthesis",
        "responses": [
            "Our voice cloning feature requires a clean 10-second audio sample of the target voice. The system then synthesizes high-fidelity matches.",
            "You can clone voices securely by uploading a high-quality WAV sample directly into our model interface."
        ]
    },
    "goodbye": {
        "label": "goodbye or exit conversation",
        "responses": [
            "Goodbye! Thank you for choosing VocloneTranslate.",
            "Have a wonderful day! Let us know if you need more cloning support."
        ]
    }
}

# Extract candidate descriptions for the model to choose from
candidate_labels = [details["label"] for details in intents_map.values()]

print("Robust NLP Classifier loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Robust NLP Classifier loaded successfully!


In [9]:
# Cell 2: Interactive Chat with Context State Memory
bot_name = "VocloneBot"
context = {}  # Tracks recent conversation states

print(f"{bot_name}: Dynamic context tracking is active. Type 'quit' to exit.")
print("-" * 60)

while True:
    user_input = input("You: ")
    if user_input.lower() == 'quit':
        print(f"{bot_name}: Goodbye!")
        break

    if not user_input.strip():
        continue

    # Perform prediction
    res = classifier(user_input, candidate_labels=candidate_labels)
    top_label = res['labels'][0]
    confidence = res['scores'][0]

    # Check threshold to verify intent confidence
    if confidence > 0.45:
        # Match the chosen label back to its tag key
        matched_tag = None
        for tag, details in intents_map.items():
            if details["label"] == top_label:
                matched_tag = tag
                break

        # Context tracking mechanics
        if matched_tag == "voice_cloning":
            context['last_action'] = "voice_cloning_info"
        elif matched_tag == "translate":
            context['last_action'] = "translation_info"

        # Output the response
        response = random.choice(intents_map[matched_tag]["responses"])
        print(f"{bot_name}: {response}")

        # Follow-up conditional logic based on tracked history
        if context.get('last_action') == "voice_cloning_info" and matched_tag != "voice_cloning":
            print(f"{bot_name}: (Pro Tip) Remember to use WAV files for the highest quality voice cloning results!")
            context.clear()  # Clear state after context-aware reminder is handled

    else:
        print(f"{bot_name}: I'm not entirely sure how to handle that statement. Could you ask me something specific about cloning or translation features?")

VocloneBot: Dynamic context tracking is active. Type 'quit' to exit.
------------------------------------------------------------
You: hi
VocloneBot: Hi there! What can I help you build or translate today?
You: build
VocloneBot: I'm not entirely sure how to handle that statement. Could you ask me something specific about cloning or translation features?
You: translate
VocloneBot: VocloneTranslate allows you to seamlessly translate audio and text across 20+ languages while preserving speech context.


KeyboardInterrupt: Interrupted by user

In [10]:
%%writefile app.py
import streamlit as st
from transformers import pipeline
import random

# Set up page configuration
st.set_page_config(page_title="VocloneTranslate Support", page_icon="🤖", layout="centered")

# Cache the classifier model loading so it doesn't reload on every interaction
@st.cache_resource
def load_classifier():
    return pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

classifier = load_classifier()

# Intent structures mapping back to the VocloneTranslate workspace
intents_map = {
    "greeting": {
        "label": "greeting or hello",
        "responses": [
            "Hello! Welcome to VocloneTranslate Support. How can I assist you with voice cloning or translation today?",
            "Hi there! What can I help you build or translate today?"
        ]
    },
    "translate": {
        "label": "language translation or audio translation text features",
        "responses": [
            "VocloneTranslate allows you to seamlessly translate audio and text across 20+ languages while preserving speech context.",
            "To translate, simply upload your source file, choose your target language, and hit Translate!"
        ]
    },
    "voice_cloning": {
        "label": "voice cloning or copying voice speech synthesis",
        "responses": [
            "Our voice cloning feature requires a clean 10-second audio sample of the target voice. The system then synthesizes high-fidelity matches.",
            "You can clone voices securely by uploading a high-quality WAV sample directly into our model interface."
        ]
    },
    "goodbye": {
        "label": "goodbye or exit conversation",
        "responses": [
            "Goodbye! Thank you for choosing VocloneTranslate.",
            "Have a wonderful day! Let us know if you need more cloning support."
        ]
    }
}

candidate_labels = [details["label"] for details in intents_map.values()]

# Title and Description
st.title("🤖 VocloneTranslate AI Assistant")
st.caption("A multi-turn intelligent agent for intent recognition and customer assistance[cite: 5, 29].")

# Initialize persistent session state memory for multi-turn tracking
if "messages" not in st.session_state:
    st.session_state.messages = [{"role": "assistant", "content": "Hello! I am your VocloneTranslate virtual assistant. Ask me anything about cloning voices or translation tools!"}]

if "context" not in st.session_state:
    st.session_state.context = {}

# Display conversation history
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# Handle user interaction
if user_input := st.chat_input("Type your message here..."):
    # Render user chat bubble
    with st.chat_message("user"):
        st.write(user_input)
    st.session_state.messages.append({"role": "user", "content": user_input})

    # Run intent classification
    res = classifier(user_input, candidate_labels=candidate_labels)
    top_label = res['labels'][0]
    confidence = res['scores'][0]

    # Process intent resolution and map tag
    if confidence > 0.45:
        matched_tag = None
        for tag, details in intents_map.items():
            if details["label"] == top_label:
                matched_tag = tag
                break

        # Context tracking mechanics
        if matched_tag == "voice_cloning":
            st.session_state.context['last_action'] = "voice_cloning_info"
        elif matched_tag == "translate":
            st.session_state.context['last_action'] = "translation_info"

        bot_reply = random.choice(intents_map[matched_tag]["responses"])

        # Follow-up conditional check using historical session context
        if st.session_state.context.get('last_action') == "voice_cloning_info" and matched_tag != "voice_cloning":
            bot_reply += "\n\n*(Pro Tip: Remember to upload uncompressed 16-bit WAV audio files for high-fidelity cloning!)*"
            st.session_state.context.clear()

    else:
        bot_reply = "I'm not entirely sure how to handle that statement. Could you ask me something specific about cloning or translation features?"

    # Render assistant chat bubble
    with st.chat_message("assistant"):
        st.write(bot_reply)
    st.session_state.messages.append({"role": "assistant", "content": bot_reply})

Writing app.py


In [12]:
# Cell 4 (Fixed): Expose and Run with CORS/XSRF Overrides
st_config = [
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false"
]

# Print your public IP address for authentication
print("Your Public IP Address is:")
!curl ipv4.icanhazip.com

# Boot up the server in the background with the security overrides
import subprocess
subprocess.Popen(st_config)

# Restart localtunnel
!npx localtunnel --port 8501

Your Public IP Address is:
34.125.22.64
⠙⠹⠸⠼⠴⠦⠧your url is: https://eight-shoes-deny.loca.lt
^C
